In [1]:
from google.colab import drive
from google.colab import output
import os

drive.mount('/content/drive')

notebook_name = 'Demand_Forecasting_EDA_Pipeline.ipynb'
notebook_path = os.path.join('/content/drive/MyDrive/Ai demand restaurnat and wastage forcasring', notebook_name)


MessageError: Failed to issue request POST https://colab.research.google.com/tun/m/credentials-propagation/m-s-kkb-ass1a2-1xa17cqclrs09?authtype=dfs_ephemeral&version=2&dryrun=false&propagate=true&record=false&authuser=0: Bad Request
Response body: 
<!DOCTYPE html>
<html lang=en>
  <meta charset=utf-8>
  <meta name=viewport content="initial-scale=1, minimum-scale=1, width=device-width">
  <title>Error 400 (Bad Request)!!1</title>
  <style>
    *{margin:0;padding:0}html,code{font:15px/22px arial,sans-serif}html{background:#fff;color:#222;padding:15px}body{margin:7% auto 0;max-width:390px;min-height:180px;padding:30px 0 15px}* > body{background:url(//www.google.com/images/errors/robot.png) 100% 5px no-repeat;padding-right:205px}p{margin:11px 0 22px;overflow:hidden}ins{color:#777;text-decoration:none}a img{border:0}@media screen and (max-width:772px){body{background:none;margin-top:0;max-width:none;padding-right:0}}#logo{background:url(//www.google.com/images/logos/errorpage/error_logo-15

In [ ]:
# ==============================
# 🚀 WEEK 1: DEMAND FORECASTING - EDA PIPELINE
# ==============================

# ==============================
# 📦 INSTALL & IMPORTS
# ==============================
!pip install -q kagglehub plotly statsmodels seaborn

import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

import warnings
warnings.filterwarnings("ignore")

# ==============================
# 📂 LOAD DATA
# ==============================
path = kagglehub.dataset_download("kannanaikkal/food-demand-forecasting")
print("Dataset path:", path)

import os
files = os.listdir(path)
print("Files:", files)

# Load main dataset (adjust if needed)
df = pd.read_csv(os.path.join(path, "train.csv"))

print("\nDataset Shape:", df.shape)
df.head()

In [ ]:
# ==============================
# 🧹 DATA CLEANING
# ==============================

df.info()

# Convert date
df['week'] = pd.to_datetime(df['week'])

# Rename for clarity
df.rename(columns={'week': 'date'}, inplace=True)

# Check missing values
print("\nMissing Values:\n", df.isnull().sum())

# Drop duplicates
df = df.drop_duplicates()

# Sort by date
df = df.sort_values(by='date')

df.head()

In [ ]:
# ==============================
# 📊 FEATURE ENGINEERING
# ==============================

df['day_of_week'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month
df['year'] = df['date'].dt.year
df['is_weekend'] = df['day_of_week'].isin([5,6]).astype(int)

df.head()

In [ ]:
# ==============================
# 📈 AGGREGATION (TIME SERIES)
# ==============================

daily_sales = df.groupby('date')['num_orders'].sum().reset_index()

daily_sales = daily_sales.set_index('date')

daily_sales.head()

In [ ]:
# ==============================
# 📊 TREND ANALYSIS
# ==============================

plt.figure(figsize=(14,5))
plt.plot(daily_sales, label='Daily Demand')
plt.title("📈 Demand Over Time")
plt.legend()
plt.show()

In [ ]:
# ==============================
# 📊 ROLLING MEAN
# ==============================

daily_sales['rolling_7'] = daily_sales['num_orders'].rolling(7).mean()
daily_sales['rolling_14'] = daily_sales['num_orders'].rolling(14).mean()

plt.figure(figsize=(14,5))
plt.plot(daily_sales['num_orders'], label='Actual')
plt.plot(daily_sales['rolling_7'], label='7-day Avg')
plt.plot(daily_sales['rolling_14'], label='14-day Avg')
plt.legend()
plt.title("📊 Rolling Mean Analysis")
plt.show()

In [ ]:
# ==============================
# 📊 ROLLING MEAN
# ==============================

daily_sales['rolling_7'] = daily_sales['num_orders'].rolling(7).mean()
daily_sales['rolling_14'] = daily_sales['num_orders'].rolling(14).mean()

plt.figure(figsize=(14,5))
plt.plot(daily_sales['num_orders'], label='Actual')
plt.plot(daily_sales['rolling_7'], label='7-day Avg')
plt.plot(daily_sales['rolling_14'], label='14-day Avg')
plt.legend()
plt.title("📊 Rolling Mean Analysis")
plt.show()

In [ ]:
# ==============================
# 📆 WEEKDAY ANALYSIS
# ==============================

weekday_sales = df.groupby('day_of_week')['num_orders'].mean()

plt.figure(figsize=(8,5))
sns.barplot(x=weekday_sales.index, y=weekday_sales.values)
plt.title("📆 Avg Demand by Day of Week")
plt.show()

In [ ]:
# ==============================
# 📅 MONTHLY ANALYSIS
# ==============================

monthly_sales = df.groupby('month')['num_orders'].mean()

plt.figure(figsize=(8,5))
sns.barplot(x=monthly_sales.index, y=monthly_sales.values)
plt.title("📅 Avg Demand by Month")
plt.show()

In [ ]:
# ==============================
# 🍽️ TOP MEALS
# ==============================

top_meals = df.groupby('meal_id')['num_orders'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(10,5))
top_meals.plot(kind='bar')
plt.title("🍽️ Top 10 Meals by Demand")
plt.show()

In [ ]:
# ==============================
# 💰 PRICE VS DEMAND
# ==============================

plt.figure(figsize=(8,5))
sns.scatterplot(data=df, x='checkout_price', y='num_orders')
plt.title("💰 Price vs Demand")
plt.show()

print("Correlation:\n", df[['checkout_price','num_orders']].corr())

In [ ]:
# ==============================
# 📉 TIME SERIES DECOMPOSITION
# ==============================

decomposition = seasonal_decompose(daily_sales['num_orders'], model='additive', period=7)

decomposition.plot()
plt.show()

In [ ]:
# ==============================
# 🔁 AUTOCORRELATION
# ==============================

plt.figure(figsize=(10,4))
plot_acf(daily_sales['num_orders'], lags=30)
plt.show()

plt.figure(figsize=(10,4))
plot_pacf(daily_sales['num_orders'], lags=30)
plt.show()

In [ ]:
# ==============================
# 🔥 CORRELATION HEATMAP
# ==============================

plt.figure(figsize=(10,6))
# Select only numeric columns for correlation calculation
sns.heatmap(df.select_dtypes(include=np.number).corr(), annot=True, cmap='coolwarm')
plt.title("🔥 Feature Correlation Heatmap")
plt.show()

In [ ]:
# ==============================
# 🧠 BUSINESS INSIGHTS (PRINT)
# ==============================

print("📊 INSIGHTS:")
print("- Demand shows clear time-based patterns.")
print("- Weekly seasonality likely present (check weekday chart).")
print("- Top meals contribute majority of revenue.")
print("- Price impacts demand (see correlation).")
print("- Rolling averages smooth short-term fluctuations.")
print("- Autocorrelation shows past demand influences future demand.")

In [ ]:
from statsmodels.tsa.seasonal import STL

# Corrected: Use 'daily_sales' instead of undefined 'ts_df'
stl = STL(daily_sales['num_orders'], period=7)
res = stl.fit()
res.plot()

# manual festival boost flag
# Corrected: Use 'daily_sales' instead of undefined 'ts_df'
daily_sales['is_festival'] = daily_sales.index.strftime('%m-%d').isin(['11-12','01-14']).astype(int)

In [ ]:
!pip install ruptures
import ruptures as rpt

signal = daily_sales['num_orders'].values
model = rpt.Pelt(model="rbf").fit(signal)
breaks = model.predict(pen=10)

rpt.display(signal, breaks)

In [ ]:
# ==============================
# 📊 ADDITIONAL EDA PLOTS
# ==============================

# 1. Distribution of num_orders
plt.figure(figsize=(10, 5))
sns.histplot(df['num_orders'], bins=50, kde=True)
plt.title('Distribution of Number of Orders')
plt.xlabel('Number of Orders')
plt.ylabel('Frequency')
plt.show()



In [ ]:
# 2. Average Demand by Meal Category
category_sales = df.groupby('category')['num_orders'].mean().sort_values(ascending=False)
plt.figure(figsize=(12, 6))
sns.barplot(x=category_sales.index, y=category_sales.values, palette='viridis')
plt.title('Average Demand by Meal Category')
plt.xlabel('Meal Category')
plt.ylabel('Average Number of Orders')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:

# 3. Average Demand by Cuisine
cuisine_sales = df.groupby('cuisine')['num_orders'].mean().sort_values(ascending=False)
plt.figure(figsize=(10, 5))
sns.barplot(x=cuisine_sales.index, y=cuisine_sales.values, palette='magma')
plt.title('Average Demand by Cuisine')
plt.xlabel('Cuisine Type')
plt.ylabel('Average Number of Orders')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()



In [ ]:
# 4. Average Demand by Center Type
center_type_sales = df.groupby('center_type')['num_orders'].mean().sort_values(ascending=False)
plt.figure(figsize=(8, 5))
sns.barplot(x=center_type_sales.index, y=center_type_sales.values, palette='cividis')
plt.title('Average Demand by Center Type')
plt.xlabel('Center Type')
plt.ylabel('Average Number of Orders')
plt.tight_layout()
plt.show()

In [ ]:


# 5. Top 10 Centers by Demand
top_centers = df.groupby('center_id')['num_orders'].sum().sort_values(ascending=False).head(10)
plt.figure(figsize=(12, 6))
sns.barplot(x=top_centers.index, y=top_centers.values, palette='plasma')
plt.title('Top 10 Centers by Total Demand')
plt.xlabel('Center ID')
plt.ylabel('Total Number of Orders')
plt.tight_layout()
plt.show()



In [ ]:
# 6. Demand vs. Emailer for Promotion
emailer_demand = df.groupby('emailer_for_promotion')['num_orders'].mean()
plt.figure(figsize=(7, 4))
sns.barplot(x=emailer_demand.index, y=emailer_demand.values, palette='viridis')
plt.title('Average Demand with/without Emailer Promotion (0=No, 1=Yes)')
plt.xlabel('Emailer for Promotion')
plt.ylabel('Average Number of Orders')
plt.xticks([0, 1], ['No', 'Yes'])
plt.tight_layout()
plt.show()


In [ ]:

# 7. Demand vs. Homepage Featured
homepage_demand = df.groupby('homepage_featured')['num_orders'].mean()
plt.figure(figsize=(7, 4))
sns.barplot(x=homepage_demand.index, y=homepage_demand.values, palette='coolwarm')
plt.title('Average Demand with/without Homepage Featured (0=No, 1=Yes)')
plt.xlabel('Homepage Featured')
plt.ylabel('Average Number of Orders')
plt.xticks([0, 1], ['No', 'Yes'])
plt.tight_layout()
plt.show()


In [ ]:
# ==============================
# 🚀 WEEK 1: GRID EDA DASHBOARD (ONE CELL)
# ==============================

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

from statsmodels.tsa.seasonal import STL
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Set up a single figure for all plots with an appropriate size for a 4x3 grid
plt.figure(figsize=(22, 24))

# ==============================
# PREP
# ==============================
# The main 'df' DataFrame is already prepared in previous steps (date conversion, feature engineering)
# No need for df.copy(), re-converting date, or re-adding day_of_week/month to 'df' here.

# Aggregate daily demand
ts = df.groupby('date')['num_orders'].sum().reset_index()
ts.set_index('date', inplace=True)

# Features for 'ts' (time series dataframe)
ts['rolling_7'] = ts['num_orders'].rolling(7).mean()
ts['rolling_14'] = ts['num_orders'].rolling(14).mean()

# ==============================
# 1. TREND: Demand Over Time
# ==============================
plt.subplot(4,3,1)
plt.plot(ts['num_orders'], color='darkblue', linewidth=1.5)
plt.title("1. Trend: Demand Over Time", fontsize=14)
plt.xlabel("Date", fontsize=12)
plt.ylabel("Number of Orders", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)

# ==============================
# 2. ROLLING MEAN Analysis
# ==============================
plt.subplot(4,3,2)
plt.plot(ts['num_orders'], label='Actual', color='gray', alpha=0.7)
plt.plot(ts['rolling_7'], label='7-day Avg', color='darkorange', linewidth=2)
plt.plot(ts['rolling_14'], label='14-day Avg', color='green', linewidth=2)
plt.legend()
plt.title("2. Rolling Mean Analysis", fontsize=14)
plt.xlabel("Date", fontsize=12)
plt.ylabel("Number of Orders", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)

# ==============================
# 3. WEEKDAY PATTERN: Avg Demand by Day of Week
# ==============================
plt.subplot(4,3,3)
# Ensure 'day_of_week' is available from the pre-processed df
weekday = df.groupby('day_of_week')['num_orders'].mean()
sns.barplot(x=weekday.index, y=weekday.values, palette='viridis')
plt.title("3. Avg Demand by Weekday", fontsize=14)
plt.xlabel("Day of Week (0=Mon, 6=Sun)", fontsize=12)
plt.ylabel("Average Number of Orders", fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# ==============================
# 4. MONTHLY PATTERN: Avg Demand by Month
# ==============================
plt.subplot(4,3,4)
# Ensure 'month' is available from the pre-processed df
monthly = df.groupby('month')['num_orders'].mean()
sns.barplot(x=monthly.index, y=monthly.values, palette='plasma')
plt.title("4. Avg Demand by Month", fontsize=14)
plt.xlabel("Month", fontsize=12)
plt.ylabel("Average Number of Orders", fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# ==============================
# 5. TOP MEALS by Demand
# ==============================
plt.subplot(4,3,5)
top_meals = df.groupby('meal_id')['num_orders'].sum().sort_values(ascending=False).head(10)
top_meals.plot(kind='bar', color='skyblue')
plt.title("5. Top 10 Meals by Total Demand", fontsize=14)
plt.xlabel("Meal ID", fontsize=12)
plt.ylabel("Total Number of Orders", fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)

# ==============================
# 6. PRICE VS DEMAND
# ==============================
plt.subplot(4,3,6)
sns.scatterplot(data=df.sample(5000, random_state=42), x='checkout_price', y='num_orders', alpha=0.5, color='darkred')
plt.title("6. Price vs Demand (Sampled)", fontsize=14)
plt.xlabel("Checkout Price", fontsize=12)
plt.ylabel("Number of Orders", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)

# ==============================
# 7. STL DECOMPOSITION: Trend Component
# ==============================
plt.subplot(4,3,7)
stl = STL(ts['num_orders'], period=7)
res = stl.fit()
plt.plot(res.trend, color='blue', linewidth=1.5)
plt.title("7. STL Decomposition: Trend Component", fontsize=14)
plt.xlabel("Date", fontsize=12)
plt.ylabel("Trend", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)

# ==============================
# 8. STL DECOMPOSITION: Seasonal Component
# ==============================
plt.subplot(4,3,8)
plt.plot(res.seasonal, color='purple', linewidth=1.5)
plt.title("8. STL Decomposition: Seasonal Component", fontsize=14)
plt.xlabel("Date", fontsize=12)
plt.ylabel("Seasonality", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)

# ==============================
# 9. Demand Distribution
# ==============================
plt.subplot(4,3,9)
sns.histplot(ts['num_orders'], bins=30, kde=True, color='teal')
plt.title("9. Daily Demand Distribution", fontsize=14)
plt.xlabel("Number of Orders (Daily Sum)", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# ==============================
# 10. AUTOCORRELATION (ACF)
# ==============================
plt.subplot(4,3,10)
plot_acf(ts['num_orders'], lags=30, ax=plt.gca(), title="10. Autocorrelation (ACF)", color='orange')
plt.xlabel("Lags", fontsize=12)
plt.ylabel("Autocorrelation", fontsize=12)

# ==============================
# 11. PARTIAL AUTOCORRELATION (PACF)
# ==============================
plt.subplot(4,3,11)
plot_pacf(ts['num_orders'], lags=30, ax=plt.gca(), title="11. Partial Autocorrelation (PACF)", color='brown')
plt.xlabel("Lags", fontsize=12)
plt.ylabel("Partial Autocorrelation", fontsize=12)

# ==============================
# 12. FEATURE CORRELATION HEATMAP
# ==============================
plt.subplot(4,3,12)
sns.heatmap(df[['num_orders','checkout_price','base_price', 'emailer_for_promotion', 'homepage_featured']].corr(),
            annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title("12. Feature Correlation Heatmap", fontsize=14)

plt.tight_layout(pad=3.0) # Adjust padding between subplots for neatness
plt.show()


# ==============================
# 🧠 QUICK INSIGHTS (AUTO)
# ==============================

print("\n📊 QUICK INSIGHTS:")
print(f"- Avg Daily Demand: {ts['num_orders'].mean():.2f}")
print(f"- Peak Demand: {ts['num_orders'].max()}")
print(f"- Lowest Demand: {ts['num_orders'].min()}")
print("- Weekly patterns visible (see weekday chart)")
print("- Rolling mean smooths fluctuations")
print("- Price impacts demand (scatter)")
print("- Autocorrelation shows dependence on past demand")


In [ ]:
# ==============================
# 🚀 ADVANCED GRID EDA DASHBOARD (12 PANELS)
# ==============================

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

from statsmodels.tsa.seasonal import STL
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

plt.figure(figsize=(22,16))

# ==============================
# PREP
# ==============================
df = df.copy()
df['date'] = pd.to_datetime(df['date'])

ts = df.groupby('date')['num_orders'].sum().reset_index()
ts.set_index('date', inplace=True)

ts['rolling_7'] = ts['num_orders'].rolling(7).mean()
ts['rolling_14'] = ts['num_orders'].rolling(14).mean()

df['day_of_week'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month

# ==============================
# 1. TREND
# ==============================
plt.subplot(4,3,1)
plt.plot(ts['num_orders'])
plt.title("Trend Over Time")

# ==============================
# 2. ROLLING
# ==============================
plt.subplot(4,3,2)
plt.plot(ts['num_orders'], label='Actual')
plt.plot(ts['rolling_7'], label='7-day')
plt.plot(ts['rolling_14'], label='14-day')
plt.legend()
plt.title("Rolling Trends")

# ==============================
# 3. DISTRIBUTION
# ==============================
plt.subplot(4,3,3)
sns.histplot(ts['num_orders'], kde=True)
plt.title("Demand Distribution")

# ==============================
# 4. WEEKDAY
# ==============================
plt.subplot(4,3,4)
weekday = df.groupby('day_of_week')['num_orders'].mean()
sns.barplot(x=weekday.index, y=weekday.values)
plt.title("Weekday Demand")

# ==============================
# 5. MONTH
# ==============================
plt.subplot(4,3,5)
monthly = df.groupby('month')['num_orders'].mean()
sns.barplot(x=monthly.index, y=monthly.values)
plt.title("Monthly Demand")

# ==============================
# 6. BOXPLOT (OUTLIERS)
# ==============================
plt.subplot(4,3,6)
sns.boxplot(y=ts['num_orders'])
plt.title("Outliers Detection")

# ==============================
# 7. TOP MEALS
# ==============================
plt.subplot(4,3,7)
top_meals = df.groupby('meal_id')['num_orders'].sum().nlargest(10)
top_meals.plot(kind='bar')
plt.title("Top Meals")

# ==============================
# 8. PRICE VS DEMAND
# ==============================
plt.subplot(4,3,8)
sns.scatterplot(data=df.sample(5000), x='checkout_price', y='num_orders', alpha=0.5)
plt.title("Price vs Demand")

# ==============================
# 9. STL TREND
# ==============================
plt.subplot(4,3,9)
stl = STL(ts['num_orders'], period=7)
res = stl.fit()
plt.plot(res.trend)
plt.title("STL Trend")

# ==============================
# 10. AUTOCORRELATION
# ==============================
plt.subplot(4,3,10)
plot_acf(ts['num_orders'], lags=30, ax=plt.gca())
plt.title("ACF")

# ==============================
# 11. PACF
# ==============================
plt.subplot(4,3,11)
plot_pacf(ts['num_orders'], lags=30, ax=plt.gca())
plt.title("PACF")

# ==============================
# 12. CORRELATION HEATMAP
# ==============================
plt.subplot(4,3,12)
sns.heatmap(df[['num_orders','checkout_price','base_price']].corr(), annot=True, cmap='coolwarm')
plt.title("Feature Correlation")

plt.tight_layout()
plt.show()

# ==============================
# 🧠 SMART INSIGHTS
# ==============================

print("\n📊 SMART INSIGHTS:")
print(f"• Avg Demand: {ts['num_orders'].mean():.2f}")
print(f"• Peak Demand: {ts['num_orders'].max()}")
print(f"• Volatility (Std): {ts['num_orders'].std():.2f}")

# trend insight
trend_slope = np.polyfit(range(len(ts)), ts['num_orders'], 1)[0]
print("• Trend:", "Increasing 📈" if trend_slope > 0 else "Decreasing 📉")

# price correlation
corr = df[['checkout_price','num_orders']].corr().iloc[0,1]
print(f"• Price vs Demand Correlation: {corr:.2f}")

if corr < 0:
    print("  → Discounts increase demand 🔥")
else:
    print("  → Price not strongly affecting demand")

print("• Weekly patterns exist (check weekday plot)")
print("• Strong lag dependency (see ACF/PACF)")